In [ ]:
cd ../..

## Feature Ideas

- For the following time periods:
    - All time
    - Prev 3 months
    - 3 - 6 months in the past
    - 6 - 9 months in the past
- Calculate the following features:
    - `weeks_since_first_review`: only for all time
    - `last_review_weeks_ago`: only for all time
    - `last_review_summary_text`: Embedding, only for all time
    - `last_reviewed_product_title`: Embedding, only for all time
    - `last_reviewed_product_category`: only for all time
    - `last_review_is_verified`: only for all time
    - `num_reviews`
    - `sum_review_ratings`
    - `avg_review_length`
    - `avg_review_rating`
    - `pct_verified_reviews`
    - `std_review_rating`
    - `min_review_rating`
    - `max_review_rating`
    - `avg_reviewed_product_rating`
    - `sum_reviewed_product_rating`
    - `std_reviewed_product_rating`
    - `min_reviewed_product_rating`
    - `max_reviewed_product_rating`
    - `avg_reviewed_product_price`
    - `sum_reviewed_product_price`
    - `std_reviewed_product_price`
    - `min_reviewed_product_price`
    - `max_reviewed_product_price`
    - `reviewed_product_modal_category`

In [ ]:
import duckdb
from torch_frame.utils import infer_df_stype

import utils

conn = duckdb.connect('amazon/amazon.db')
%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.displaycon=False
%config SqlMagic.autopandas=True

In [ ]:
with open('amazon/user-ltv/feats.sql', 'r') as f:
    # run once with train_labels and once with val_labels
    template = f.read()

# create train, val and test features
# takes 1 - 5 mins
for s in ['train', 'val', 'test']:
    print(f'Creating {s} table')
    query = utils.render_jinja_sql(template, dict(set=s, subsample=1_000_000))
    conn.sql(query)
    print(f'{s} table created')

In [ ]:
utils.validate_feature_tables('user_ltv', conn)

^-- Failure expected bc of subsampling

In [ ]:
%%sql train_df <<
from ltv_train_feats

In [ ]:
infer_df_stype(train_df)

In [ ]:
utils.feature_summary_df(train_df.sample(20_000), 'ltv', classification=False)

## Feature Importances

In [ ]:
import numpy as np
import shap
from torch_frame import TaskType, stype
from torch_frame.data import Dataset
from torch_frame.gbdt import LightGBM

from inferred_stypes import task_to_stypes
from train_gbdt import TASK_PARAMS

TASK = 'rel-amazon-user-ltv'

task_params = TASK_PARAMS[TASK]

In [ ]:
%%sql val_df <<
select * from user_ltv_val_feats;

In [ ]:
col_to_stype = task_to_stypes[TASK].copy()
del col_to_stype['last_review_summary_text']
del col_to_stype['last_reviewed_product_title']
val_tf = Dataset(
    val_df,
    col_to_stype=col_to_stype,
    target_col=task_params['target_col'],
).materialize().tensor_frame

In [ ]:
gbdt = LightGBM(task_type=task_params['task_type'])
gbdt.load(f'models/{TASK}_lgbm.json')
pred = gbdt.predict(tf_test=val_tf).numpy()

In [ ]:
shap.initjs()
explainer = shap.TreeExplainer(gbdt.model)

sample = np.random.randint(0, len(val_tf), size=10_000)

val_arr, _, _ = gbdt._to_lightgbm_input(val_tf[sample])
shap_values = explainer.shap_values(val_arr, pred[sample])

# TODO verify
feat_names = val_tf.col_names_dict.get(stype.categorical, []) + val_tf.col_names_dict[stype.numerical]

shap.summary_plot(shap_values, val_arr, plot_type='violin', max_display=30, feature_names=feat_names)

In [ ]:
conn.close()